# Demo 27: Exploration vs Exploitation: ε-Greedy

**挿入位置：探索と活用のスライド**　｜　**実行環境：Google Colab**

正解のわからない3つの選択肢で、現在の最善手を選ぶことと、新しい選択肢を試すことのバランスを体験します。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
ARM_NAMES=['A','B','C'];TRUE_MEANS=np.array([0.2,0.5,0.8])
def bandit_run(epsilon=.1,n=1000,seed=0):
    rng=np.random.default_rng(seed);q=np.zeros(3);cnt=np.zeros(3,dtype=int)
    actions=[];rewards=[];est=[];regret=[]
    for t in range(n):
        if rng.random()<epsilon:a=int(rng.integers(3))
        else:a=int(rng.choice(np.flatnonzero(np.isclose(q,q.max()))))
        r=float(rng.normal(TRUE_MEANS[a],.6));cnt[a]+=1;q[a]+=(r-q[a])/cnt[a]
        actions.append(a);rewards.append(r);est.append(q.copy());regret.append(TRUE_MEANS.max()-TRUE_MEANS[a])
    return np.array(actions),np.array(rewards),np.array(est),np.cumsum(regret)

def batch_bandits(eps_list=(0,.1,.5,1),horizon=1000,repeats=50,seed=0):
    data={}
    for e in eps_list:
        rg=np.random.default_rng(seed+int(e*1000));curves=[]
        for _ in range(repeats):
            a,r,q,reg=bandit_run(e,horizon,int(rg.integers(2**30)))
            curves.append(np.cumsum(r)/np.arange(1,horizon+1))
        data[e]=np.mean(curves,axis=0)
    return data

In [ ]:
epsilon_w=widgets.FloatSlider(value=.1,min=0,max=1,step=.05,description='ε',continuous_update=False)
trials=widgets.IntSlider(value=1000,min=200,max=3000,step=200,description='Trials',continuous_update=False)
seed=widgets.IntSlider(value=10,min=0,max=100,description='Seed')
button=widgets.Button(description='Run experiment',button_style='primary');out=widgets.Output()
def update(_):
    with out:
        clear_output(wait=True)
        act,rew,q,reg=bandit_run(epsilon_w.value,trials.value,seed.value)
        fig,ax=plt.subplots(1,3,figsize=(14,3.7))
        ax[0].bar(ARM_NAMES,np.bincount(act,minlength=3));ax[0].set(title='Actions chosen',ylabel='Count')
        ax[1].plot(q);ax[1].axhline(TRUE_MEANS.max(),ls=':',c='k',alpha=.6,label='Best true mean')
        ax[1].set(title='Estimated rewards (Q)',xlabel='Trial',ylabel='Estimated mean');ax[1].legend()
        ax[2].plot(reg);ax[2].set(title='Cumulative pseudo-regret',xlabel='Trial',ylabel='Regret')
        for a in ax:a.grid(alpha=.2)
        plt.tight_layout();plt.show()
        print('True (hidden from agent) mean rewards:',dict(zip(ARM_NAMES,TRUE_MEANS)))
        print('Estimated means:',np.round(q[-1],3),'| best-arm selection:',round(np.mean(act==2),3))
        avg=batch_bandits(horizon=trials.value,repeats=30,seed=seed.value)
        plt.figure(figsize=(9,3.5))
        for e,curve in avg.items():plt.plot(curve,label=f'ε = {e}')
        plt.title('Average reward across 30 runs');plt.xlabel('Trials');plt.ylabel('Cumulative average reward');plt.legend();plt.grid(alpha=.2);plt.show()
button.on_click(update)
display(widgets.VBox([widgets.HBox([epsilon_w,trials,seed]),button,out]));update(None)

### ε-greedyでの割合
確率 $1-arepsilon$ で最善と推定した行動を選び、確率 $arepsilon$ で**全行動から一様ランダム**に選びます。したがって、ランダム選択の中には最善の行動も含まれます（それを除外するわけではありません）。

**安全性**：このデモは安全な架空のバンディットです。実際のロボットで無制限にランダムな関節トルクを試すことは推奨しません。

## Try it yourself / 実験課題
- ε=0でたまたま不利な行動に固定される例を探す。
- ε=0.1とε=1で平均報酬を比較する。
- 最善行動を選ぶ割合が厳密には1−εより大きくなり得る理由を説明する。